---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** ___육건우___ &nbsp; **학번:** _____20251284_____ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

나의 탐구 주제는 "새 화학물질의 발암 가능성을 자동 예측하는 모델 만들기"

In [13]:
import pandas as pd

file_path = 'cancer.xlsx' 
sheet_all = 'Data'

# pd.read_excel 함수를 사용해 특정 시트를 읽어온다.
df = pd.read_excel(file_path, sheet_name=sheet_all)

print(df.shape) #전체 데이터의 크기 행과 열의 개수 확인
print(df.columns) #전체 컬럼 이름 확인

(10351, 28)
Index(['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name',
       'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient',
       'Mixture', 'Species', 'Sex', 'Strain', 'Route', 'Level_of_Evidence',
       'Tissue', 'Location', 'Lesion', 'Assay', 'Endpoint', 'Response',
       'Response_Unit', 'Reference', 'URL', 'SMILES', 'Preferred_Name',
       'Synonyms', 'URL_CompTox', 'URL_CEBS'],
      dtype='object')


In [16]:
# 전체 데이터 중 'NTP Carcinogenicity' 관련 데이터만 골라내기
is_ntp = df["Endpoint"] == "Level of evidence of carcinogenic activity"
ntp_df = df[is_ntp].copy()

# 그중에서 생체 내(In Vivo) 실험 데이터만 남기기
ntp_in_vivo_df = ntp_df[ntp_df["Data_Type"] == "In Vivo"].copy()

# 분자 구조(SMILES)가 비어있는 행은 제외하기
ntp_df = ntp_df[ntp_df["SMILES"].notna()]

# 현재까지 정제된 데이터의 크기 확인
ntp_in_vivo_df.shape

(2343, 28)

In [17]:
# NTP Carcinogenicity + In Vivo 조건 만족하는 데이터의 종별 개수 확인
species_counts_final = ntp_in_vivo_df["Species"].value_counts()

# 결과 확인하기
species_counts_final

Species
Rat        1167
Mouse      1166
Hamster      10
Name: count, dtype: int64

In [18]:
# 랫(Rat) 데이터의 성별 구성 확인하기
rat_sex = ntp_in_vivo_df[ntp_in_vivo_df["Species"] == "Rat"]["Sex"].value_counts()
print("--- 랫(Rat) 성별 분포 ---")
print(rat_sex)

# 마우스(Mouse) 데이터의 성별 구성 확인하기
mouse_sex = ntp_in_vivo_df[ntp_in_vivo_df["Species"] == "Mouse"][
    "Sex"
].value_counts()
print("--- 마우스(Mouse) 성별 분포 ---")
print(mouse_sex)

--- 랫(Rat) 성별 분포 ---
Sex
Female    584
Male      583
Name: count, dtype: int64
--- 마우스(Mouse) 성별 분포 ---
Sex
Female    583
Male      583
Name: count, dtype: int64


In [ ]:
#=============================================================================